# AutoETS & ETS Tutorial
This tutorial explains how to use the **AutoETS** and **ETS** exponential smoothing models.

AutoETS fits an Exponential Smoothing (ETS) state-space model by **automatically selecting** the best combination of:
* **E**rror type (Additive or Multiplicative)
* **T**rend component (None, Additive, or Multiplicative)
* **S**easonal component (None, Additive, or Multiplicative)

Model selection is performed using an information criterion (AICc by default).

**ETS** is the fixed-specification variant — you choose the exact model structure (e.g. `"ANN"`, `"AAN"`) rather than letting the algorithm decide.

In [ ]:
import sys
import os

# Add parent directory to path so we can import the library modules
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import jax
import jax.numpy as jnp
import numpy as np

from auto_ets import AutoETS
from ets_model import ETS
from conformal_intervals import ConformalIntervals

## Mathematical Overview

ETS models decompose a time series into **level** ($\ell_t$), **trend** ($b_t$), and **seasonal** ($s_t$) components. At each time step the model:

1. Produces a one-step-ahead forecast from the current state
2. Observes the actual value and computes the error
3. Updates the states using **smoothing parameters** ($\alpha$, $\beta$, $\gamma$, $\phi$)

### Example: Additive Error, Additive Trend, No Seasonality (AAN)

$$\hat{y}_{t+1|t} = \ell_t + b_t$$

$$\ell_t = \ell_{t-1} + b_{t-1} + \alpha \, e_t$$

$$b_t = b_{t-1} + \beta \, e_t$$

$$e_t = y_t - \hat{y}_{t|t-1}$$

Where:
- $\alpha \in (0,1)$ controls how fast the level adapts
- $\beta \in (0,1)$ controls how fast the trend adapts
- $\phi \in [0.8, 0.98]$ is the optional damping parameter (shrinks the trend toward zero)

### Model String Notation

The three-character model string specifies **(Error, Trend, Season)**:

| Character | Error | Trend | Season |
|:---------:|:-----:|:-----:|:------:|
| **A** | Additive | Additive | Additive |
| **M** | Multiplicative | Multiplicative | Multiplicative |
| **N** | — | No trend | No season |
| **Z** | *Auto-select* | *Auto-select* | *Auto-select* |

For example, `"AAN"` = Additive error + Additive trend + No seasonality.

When you use `"ZZZ"`, AutoETS evaluates multiple combinations and picks the best one via AICc.

## When to Use

**Use AutoETS when:**
* The series may contain trend and/or seasonality
* You want automatic model selection without manual tuning
* Short to medium horizon forecasting
* You want a strong classical statistical baseline

**Avoid when:**
* Extremely short time series (< 10 observations)
* Highly irregular structural breaks or regime changes
* Complex multi-seasonality (consider TBATS instead)
* Intermittent/sparse demand (consider IMAPA or Croston instead)

## Brief Implementation Details

1. **Model selection** — AutoETS evaluates a grid of ETS configurations, each fitted via maximum likelihood, and selects the model that minimizes AICc. ETS (fixed-spec) fits exactly one user-specified structure.
2. **Parameter estimation** — Smoothing parameters and initial states are jointly optimized using `optax` (JAX-based gradient descent).
3. **Prediction intervals** — Both native Gaussian ETS intervals and conformal prediction intervals are supported.
4. **Three calling patterns** (identical API for both AutoETS and ETS):
   - **Stateful:** `fit(y)` → `predict(h)` — fit once, predict repeatedly
   - **Stateless:** `forecast(y, h)` — fit and predict in one call
   - **Forward:** `fit(y1)` → `forward(y2, h)` — reuse a fitted model's learned structure on new data

## API Contract

### Constructor

```python
AutoETS(
    season_length: int = 1,
    model: str = "ZZZ",
    damped: Optional[bool] = None,
    phi: Optional[float] = None,
    alias: str = "AutoETS",
    prediction_intervals: Optional[ConformalIntervals] = None,
)
```

**Parameters:**
- `season_length`: Periodicity of the data (e.g. 12 for monthly, 24 for hourly, 4 for quarterly)
- `model`: ETS specification string. Use `"ZZZ"` for fully automatic selection, or fix components like `"AAN"`
- `damped` / `phi`: Optional trend damping. If `damped=None`, both damped and undamped are tried
- `prediction_intervals`: Optional `ConformalIntervals` configuration for uncertainty quantification

### Methods

| Method | Description | Requires `fit()` first? |
|--------|------------|------------------------|
| `fit(y)` | Fits the optimal ETS model to training data | — |
| `predict(h, level)` | Generates h-step forecasts from the fitted state | Yes |
| `predict_in_sample(level)` | Returns in-sample fitted values for diagnostics | Yes |
| `forecast(y, h, level, fitted)` | Stateless fit + predict in one call | No |
| `forward(y, h, level, fitted)` | Apply fitted model structure to a new series | Yes |

---

## Examples

### 1. Fit and Predict — Linear Trend

We fit AutoETS on a deterministic linear trend and verify that forecasts continue the upward trajectory.

In [ ]:
# Create a simple linear trend: y = 2 + 0.5*t
a, b = 2.0, 0.5
t = jnp.arange(60, dtype=jnp.float64)
y = a + b * t

ae = AutoETS(season_length=1, model="ZZZ")
ae.fit(y)

h = 6
out = ae.predict(h=h, level=None)

mean = np.asarray(out["mean"])
last_val = float(a + b * (len(y) - 1))  # 2 + 0.5*59 = 31.5

print(f"Last training value:  {last_val}")
print(f"Expected next step:  ~{last_val + b}")
print(f"Forecast (6 steps):  {mean}")
print(f"Shape: {mean.shape}")

assert mean.shape == (h,), "Forecast shape should be (h,)"
assert np.all(np.isfinite(mean)), "All forecasts must be finite"
# The forecast should roughly continue the upward trend
assert mean[0] > last_val - 2.0, "First forecast should be near the last training value"

print("\nFit + predict on linear trend: OK")

AutoETS identified a model with a trend component and produced forecasts that continue the linear pattern. The selected model structure is stored internally and can be inspected. (If you already know the right model, use `ETS(model="AAN")` instead — see the ETS section below.)

In [ ]:
# Inspect the selected model
print("Selected model:", ae.model_.get("method", "N/A"))
print("Components:", ae.model_.get("components", "N/A"))
print("Season length (m):", ae.model_.get("m", "N/A"))
print("Number of parameters:", ae.model_.get("n_params", "N/A"))
print("AICc:", round(ae.model_.get("aicc", float("nan")), 2))

### 2. Predict with Prediction Intervals

Prediction intervals quantify forecast uncertainty. AutoETS supports **native Gaussian intervals** computed from the state-space model's error variance.

Pass a list of `level` values (0–100) to `predict()`. Each level produces a lower (`lo-{level}`) and upper (`hi-{level}`) bound.

In [ ]:
# Predict with native ETS prediction intervals (unsorted levels on purpose)
y = jnp.asarray([10.0, 11.0, 10.5, 11.5, 12.0, 11.0, 12.5, 12.0] * 4, dtype=jnp.float64)

ae = AutoETS(season_length=1, model="ZZZ")
ae.fit(y)

out = ae.predict(h=4, level=[95, 80, 50])  # deliberately unsorted

print("Forecast mean:", np.asarray(out["mean"]))
print("Output keys:", sorted(out.keys()))

# Verify: lo <= mean <= hi for each confidence level
m = np.asarray(out["mean"])
for lv in (50, 80, 95):
    lo = np.asarray(out[f"lo-{lv}"])
    hi = np.asarray(out[f"hi-{lv}"])
    assert np.all(lo <= m + 1e-12) and np.all(m <= hi + 1e-12),         f"Mean not inside {lv}% band"
    print(f"  {lv}% band: [{lo[0]:.2f}, {hi[0]:.2f}] (step 1)")

print("\nPredict with intervals: OK — all bands are consistent")

Notice that wider confidence levels (95%) produce wider bands than narrower ones (50%), as expected. The intervals widen as we forecast further into the future.

### 3. Predict in Sample

The `predict_in_sample()` method returns **fitted values** — the model's one-step-ahead predictions for the training data. This is useful for:
- Evaluating goodness of fit
- Computing residuals ($e_t = y_t - \hat{y}_t$)
- Diagnosing model issues

When `level` is provided, it also returns fitted prediction intervals.

In [ ]:
# Predict in sample with intervals
y = jnp.asarray([10.0, 12.0, 11.0, 13.0, 12.0, 11.5, 12.5, 12.0] * 3, dtype=jnp.float64)

ae = AutoETS(season_length=1, model="ZZZ")
ae.fit(y)

res = ae.predict_in_sample(level=[80, 95])

fitted = np.asarray(res["fitted"])
lo80 = np.asarray(res["fitted-lo-80"])
lo95 = np.asarray(res["fitted-lo-95"])
hi80 = np.asarray(res["fitted-hi-80"])
hi95 = np.asarray(res["fitted-hi-95"])

print(f"Fitted values shape: {fitted.shape}")
print(f"First 5 fitted values: {fitted[:5]}")
print(f"Output keys: {sorted(res.keys())}")

# Check monotonicity: 95% band should be wider than 80%
assert np.all(lo95 <= lo80 + 1e-12), "95% lower should be <= 80% lower"
assert np.all(hi95 >= hi80 - 1e-12), "95% upper should be >= 80% upper"

# Fitted values should lie within the bands
assert np.all(lo80 <= fitted + 1e-12), "Fitted should be >= 80% lower"
assert np.all(fitted <= hi80 + 1e-12), "Fitted should be <= 80% upper"

# Compute in-sample MAE for reference
mae = float(np.mean(np.abs(fitted - np.asarray(y))))
print(f"\nIn-sample MAE: {mae:.4f}")

print("Predict in sample with interval monotonicity: OK")

### 4. Forecast (Stateless)

The `forecast()` method performs a **stateless** fit + predict in a single call. It does not require calling `fit()` first.

```python
forecast(y, h, level=None, fitted=False) -> Dict
```

This is useful for:
- Cross-validation loops where you repeatedly refit on different windows
- Parallel or distributed evaluation
- Situations where you don't need to keep model state

When `fitted=True`, the output also includes in-sample fitted values.

In [ ]:
# Stateless forecast with intervals and fitted values
a, b = 2.0, 0.5
t = jnp.arange(60, dtype=jnp.float64)
y = a + b * t

ae = AutoETS(season_length=1, model="ZZZ")

h = 5
out = ae.forecast(y=y, h=h, level=[80, 95], fitted=True)

mean = np.asarray(out["mean"])
print(f"Forecast mean: {mean}")
print(f"Output keys: {sorted(out.keys())}")

assert "mean" in out and out["mean"].shape == (h,)
assert "fitted" in out and out["fitted"].shape == y.shape

# Check that forecast intervals exist and are consistent
for lv in [80, 95]:
    assert f"lo-{lv}" in out and f"hi-{lv}" in out
    lo = np.asarray(out[f"lo-{lv}"])
    hi = np.asarray(out[f"hi-{lv}"])
    assert np.all(lo <= mean + 1e-12) and np.all(mean <= hi + 1e-12)

# Check that fitted intervals also exist
for lv in [80, 95]:
    assert f"fitted-lo-{lv}" in out and f"fitted-hi-{lv}" in out

print("\nStateless forecast with fitted values + intervals: OK")

### 5. Stateful vs Stateless — Consistency Check

The stateful path (`fit` then `predict`) and the stateless path (`forecast`) should produce identical forecasts when given the same data.

In [ ]:
# Verify stateful and stateless produce the same result
rng = np.random.RandomState(42)
y = jnp.asarray(10.0 + 0.3 * rng.randn(48), dtype=jnp.float64)

ae = AutoETS(season_length=1, model="ZZZ")

# Stateful path
ae.fit(y)
stateful = np.asarray(ae.predict(h=5, level=None)["mean"])

# Stateless path (same model, same data)
stateless = np.asarray(ae.forecast(y=y, h=5, level=None, fitted=False)["mean"])

diff = np.max(np.abs(stateful - stateless))
print(f"Stateful forecast:  {stateful}")
print(f"Stateless forecast: {stateless}")
print(f"Max |difference|:   {diff:.2e}")

assert np.allclose(stateful, stateless, atol=1e-4),     f"Stateful and stateless forecasts should match (max diff = {diff})"

print("\nStateful vs stateless consistency: OK")

### 6. Seasonal Data

AutoETS can detect and model seasonal patterns. When `season_length > 1` and the data has enough observations (at least ~2 full cycles), the model will consider seasonal ETS structures.

In [ ]:
# Quarterly seasonal data: base level + seasonal pattern + small noise
m = 4  # quarterly
base = 10.0
season = np.array([+1.0, -1.0, +2.0, -2.0], dtype=np.float64)
rng = np.random.RandomState(7)
y = jnp.asarray(
    np.tile(season, 15) + base + 0.1 * rng.randn(60),
    dtype=jnp.float64
)

ae = AutoETS(season_length=m, model="ZZZ")
ae.fit(y)

h = 8  # forecast 2 full cycles
out = ae.predict(h=h, level=None)
mean = np.asarray(out["mean"])

print(f"Selected model: {ae.model_.get('method', 'N/A')}")
print(f"Forecast (8 steps): {mean}")

assert mean.shape == (h,)
assert np.all(np.isfinite(mean)), "All forecasts must be finite"
# Forecasts should stay in the vicinity of the base level
assert np.all(np.abs(mean - base) < 5.0),     "Seasonal forecasts should be near the base level"

print("\nSeasonal data forecast: OK")

### 7. Conformal Prediction Intervals

Besides the native Gaussian intervals, AutoETS also supports **conformal prediction intervals**. These are distribution-free intervals calibrated on past forecast errors using a rolling-origin approach.

To use them, pass a `ConformalIntervals` configuration to the constructor:

```python
ConformalIntervals(
    n_windows: int,   # number of rolling windows for calibration
    h: int,           # forecast horizon used for calibration
    method: str       # "conformal_distribution" for symmetric intervals
)
```

**Important:** Conformal intervals require enough data so that the shortest training window (after removing the rolling origins) still has sufficient observations. A good rule of thumb: `len(y) - n_windows * h >= 20`.

In [ ]:
# Fit with conformal intervals (stateful path)
rng = np.random.RandomState(17)
n = 40
t_arr = np.arange(n, dtype=np.float64)
y = jnp.asarray(
    8.0 + 0.05 * t_arr + 0.5 * np.sin(2 * np.pi * t_arr / 8),
    dtype=jnp.float64
)

cfg = ConformalIntervals(n_windows=5, h=3, method="conformal_distribution")
ae = AutoETS(season_length=1, model="ZZZ", prediction_intervals=cfg)

ae.fit(y)
print("Conformity scores cached:", ae._cs is not None)

out = ae.predict(h=3, level=[80, 95])
mean = np.asarray(out["mean"])

for lv in (80, 95):
    lo = np.asarray(out[f"lo-{lv}"])
    hi = np.asarray(out[f"hi-{lv}"])
    assert np.all(lo <= mean + 1e-12) and np.all(mean <= hi + 1e-12)
    print(f"  {lv}% band (step 1): [{lo[0]:.2f}, {hi[0]:.2f}]")

print("\nConformal intervals (stateful): OK")

In [ ]:
# Conformal intervals via the stateless forecast path
rng = np.random.RandomState(19)
n = 36
t_arr = np.arange(n, dtype=np.float64)
y = jnp.asarray(
    5.0 + 0.1 * t_arr + 0.3 * np.sin(2 * np.pi * t_arr / 6),
    dtype=jnp.float64
)

cfg = ConformalIntervals(n_windows=4, h=2, method="conformal_distribution")
ae = AutoETS(season_length=1, model="ZZZ", prediction_intervals=cfg)

out = ae.forecast(y=y, h=6, level=[90], fitted=False)

mean = np.asarray(out["mean"])
lo90 = np.asarray(out["lo-90"])
hi90 = np.asarray(out["hi-90"])

assert out["mean"].shape == (6,)
assert np.all(lo90 <= mean + 1e-12) and np.all(mean <= hi90 + 1e-12)

print(f"Forecast mean: {mean}")
print(f"90% band (step 1): [{lo90[0]:.2f}, {hi90[0]:.2f}]")

print("\nConformal intervals (stateless): OK")

### 8. Error Handling

The model provides clear error messages for common misuse patterns.

In [ ]:
# Calling predict() before fit() should raise an error
ae = AutoETS(season_length=1, model="ZZZ")
try:
    ae.predict(h=3)
    print("ERROR: should have raised an exception")
except Exception as e:
    print(f"predict() before fit() raises: {type(e).__name__}: {e}")

print("\nError handling: OK")

### 9. Tiny Series Behavior

Very short time series may not have enough observations for ETS parameter estimation. The model may raise a `ValueError` or handle it gracefully, depending on the length and model complexity.

In [ ]:
# Very short series — may raise ValueError or NotImplementedError
y_tiny = jnp.asarray([10.0, 11.0, 10.5], dtype=jnp.float64)
ae = AutoETS(season_length=1, model="ZZZ")

try:
    ae.fit(y_tiny)
    out = ae.predict(h=3, level=None)
    print(f"Tiny series forecast: {np.asarray(out['mean'])}")
    print("Model handled tiny series gracefully")
except (ValueError, NotImplementedError) as e:
    print(f"Tiny series raises {type(e).__name__}: {e}")
    print("This is expected — the series is too short for ETS estimation")

print("\nTiny series handling: OK")

## Using Fit + Predict vs Forecast

| Feature | `fit()` + `predict()` | `forecast()` |
|---------|----------------------|--------------| 
| **Workflow** | Two-step: fit once, predict many times | One-step: fit + predict combined |
| **State** | Model stored in `self.model_` | No persistent state stored |
| **Best for** | Interactive exploration, inspecting model internals | Cross-validation, batch evaluation |
| **Repeated predictions** | Fast (reuses fitted model) | Re-fits each time |
| **Memory** | Stores full model state | Minimal memory footprint |

---

# ETS (Fixed Model)

The `ETS` class shares the same underlying exponential smoothing framework as AutoETS. The key difference is that **ETS uses a fixed, user-specified model structure** (e.g. `"ANN"`, `"AAN"`, `"AAA"`), whereas AutoETS automatically selects the best combination.

All model specs are supported — simple exponential smoothing (`"ANN"`), Holt's linear trend (`"AAN"`), additive seasonality (`"AAA"`), and multiplicative variants. ETS exposes the same methods as AutoETS: `fit`, `predict`, `predict_in_sample`, `forecast`, and `forward`.

### When to use ETS vs AutoETS

| Scenario | Use |
|----------|-----|
| Don't know the right model | **AutoETS** (`model="ZZZ"`) |
| Want to enforce a specific structure | **ETS** (e.g. `model="AAN"`) |
| Need reproducibility with a known spec | **ETS** |
| Exploratory analysis | **AutoETS** |

### ETS API Contract

```python
ETS(
    season_length: int = 1,
    model: str = "ANN",
    damped: Optional[bool] = None,
    phi: Optional[float] = None,
    max_iter: Optional[int] = None,
    alias: str = "ETS",
    prediction_intervals: Optional[ConformalIntervals] = None,
)
```

- `model`: Fixed ETS specification string (e.g. `"ANN"` for simple exponential smoothing, `"AAN"` for Holt's linear trend, `"AAA"` for additive seasonality)
- `max_iter`: Number of optimization steps. If `None`, a sensible default is computed based on the data length and model complexity.
- All other parameters are the same as AutoETS

**Methods:** `fit`, `predict`, `predict_in_sample`, `forecast`, `forward` — the interface is identical to AutoETS, so any code using AutoETS can switch to ETS by replacing the constructor.

### ETS Example: Fit + Predict with Linear Trend (AAN)

We fit ETS with an explicit `"AAN"` specification (Additive error, Additive trend, No seasonality) on a linear trend with noise. Since the model spec is fixed, ETS skips the model-selection grid and fits exactly one structure — making it faster than AutoETS when you already know the right model.

In [ ]:
# ETS with fixed AAN model (Additive error, Additive trend, No seasonality)
a, b = 2.0, 0.5
rng = np.random.RandomState(42)
t = np.arange(80, dtype=np.float64)
y = jnp.asarray(a + b * t + 0.3 * rng.randn(80), dtype=jnp.float64)

et = ETS(season_length=1, model="AAN", max_iter=200)
et.fit(y)

h = 6
out = et.predict(h=h, level=[80, 95])
mean = np.asarray(out["mean"])

last_val = float(y[-1])
print(f"Last training value:  {last_val:.2f}")
print(f"Forecast (6 steps):   {mean}")
print(f"Selected model:       {et.model_.get('method', 'N/A')}")
print(f"Output keys:          {sorted(out.keys())}")

assert mean.shape == (h,)
assert np.all(np.isfinite(mean)), "All forecasts must be finite"
assert mean[0] > last_val - 5.0, "Forecast should be near the last training value"

# Verify intervals
for lv in [80, 95]:
    lo = np.asarray(out[f"lo-{lv}"])
    hi = np.asarray(out[f"hi-{lv}"])
    assert np.all(lo <= mean + 1e-12) and np.all(mean <= hi + 1e-12)

print("\nETS fit + predict (AAN): OK")

### ETS Stateless Forecast (AAN)

Just like AutoETS, `ETS.forecast()` performs a stateless fit + predict in one call. When `fitted=True`, it also returns in-sample fitted values, which is useful for diagnostics and cross-validation loops.

In [ ]:
# ETS stateless forecast with fitted values (AAN)
rng2 = np.random.RandomState(33)
t2 = np.arange(60, dtype=np.float64)
y2 = jnp.asarray(5.0 + 0.2 * t2 + 0.3 * rng2.randn(60), dtype=jnp.float64)

et = ETS(season_length=1, model="AAN", max_iter=200)
out = et.forecast(y=y2, h=4, level=[80], fitted=True)

print(f"ETS Forecast: {np.asarray(out['mean'])}")
print(f"Fitted shape: {out['fitted'].shape}")
print(f"Output keys:  {sorted(out.keys())}")

assert "mean" in out and out["mean"].shape == (4,)
assert "fitted" in out and out["fitted"].shape == y2.shape

print("\nETS stateless forecast (AAN): OK")

## Edge Cases and Limitations

### Edge Cases

1. **Short series:** Both AutoETS and ETS require at least ~6–10 observations depending on the model complexity. Series shorter than `n_params + 4` raise `NotImplementedError`.

2. **Conformal intervals:** Require enough data so that even the shortest rolling-origin window has sufficient observations. Rule of thumb: `len(y) - n_windows * h >= 20`.

3. **`phi` range:** If specified, `phi` must be a float in $[0.8, 0.98]$.

4. **Multiplicative models:** Require strictly positive data. AutoETS automatically avoids multiplicative structures on non-positive series.

5. **Seasonal models:** Seasonal components (`"A"` or `"M"` in the third position) require `season_length > 1` and at least ~2 full cycles of data.

### Limitations

1. **Automatic selection may oversimplify:** On noisy data with weak patterns, AutoETS may select `ANN` (simple exponential smoothing) even when trend or seasonality is present.

2. **Gradual change assumption:** ETS assumes level, trend, and seasonality change smoothly — sudden structural breaks can cause poor forecasts.

3. **Single seasonality only:** ETS supports one seasonal period. For multi-seasonal data (e.g. daily + weekly), consider TBATS.

4. **Optimization sensitivity:** The gradient-based optimizer may converge to a local minimum. Increasing `max_iter` can help in challenging cases.